In [1]:
import pandas as pd
import requests
import geopandas as gpd
from rasterstats import zonal_stats
import os, glob
import rasterio
import numpy as np
from rasterio.features import rasterize
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

raster_folder = "./data"

In [2]:
def get_key_from_environment(file_path: str, key: str) -> str | None:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")

In [4]:
def get_data(scale):
    header = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


    start_date = datetime(2020, 9, 1)   # Sept 2024
    end_date = datetime(2025, 8, 1)     # Aug 2025

    # Loop over months
    date = start_date
    while date <= end_date:
        date_str = date.strftime("%Y-%m")
        url = f"https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale{scale:03d}&date={date_str}"
        res = requests.get(url, headers=header)
            
        if res.status_code == 200:
            file = f"./data/spi{scale:03d}_{date_str}.tif"
            with open(file, "wb") as f:
                f.write(res.content)
            
        # Move to next month
        date += relativedelta(months=1)

In [ ]:
def get_averages(scale, division, id_col):
shp_path = f"../public/shapefiles/{division}.shp"

# Load shapefile
gdf = gpd.read_file(shp_path)
gdf = gdf.reset_index(drop=True)

# --- Handle duplicates depending on division type ---
if division in ["county", "climate"]:
    # Dissolve by id_col (merging same-name geometries into one)
    gdf = gdf.dissolve(by=id_col, as_index=False)
else:
    # Make IDs unique by adding -1, -2, etc. for duplicates
    counts = {}
    unique_names = []
    for name in gdf[id_col]:
        if name in counts:
            counts[name] += 1
            unique_names.append(f"{name}-{counts[name]}")
        else:
            counts[name] = 0
            unique_names.append(name)
    gdf[id_col] = unique_names

# Collect rasters
tifs = sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}_*.tif")))
if not tifs:
    raise FileNotFoundError("No rasters found")

# Get raster metadata
with rasterio.open(tifs[0]) as src:
    raster_crs = src.crs
    transform = src.transform
    shape = (src.height, src.width)

if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)

# Rasterize polygons once
shapes = [(geom, idx) for idx, geom in enumerate(gdf.geometry)]
mask = rasterize(shapes, out_shape=shape, transform=transform, fill=-1, dtype="int32")

records = []

for tif in tifs:
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    with rasterio.open(tif) as src:
        arr = src.read(1, masked=True)

    for idx, div in enumerate(gdf[id_col]):
        poly_mask = mask == idx
        vals = arr[poly_mask]

        # Drop masked values cleanly
        if np.ma.is_masked(vals):
            vals = vals.compressed()
        mean_val = np.nan if vals.size == 0 else np.nanmean(vals)
        records.append({"division": div, "date": date, "mean_spi": mean_val})

df = pd.DataFrame(records)

df = (
    df.groupby(["division", "date"])["mean_spi"]
    .mean()
    .reset_index()
)

df = df.pivot(index="division", columns="date", values="mean_spi")
df = df.reindex(sorted(df.columns), axis=1)

out_csv = f"../public/{division}_spi{scale}.csv"
df.to_csv(out_csv)
print(f"Saved {out_csv}")


Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-01 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-02 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-03 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-04 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-05 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-06 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-07 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-08 ...
Fetching https://api.hcdp.ikewai.org/raster?datatype=rainfall&production=new&period=month&date=1990-09 ...
Fetching https://api.hcdp.ikewai.org/

In [11]:
#statewide
shapefile = "../public/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"rainfall_*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    with rasterio.open(tif) as src:
        nodata = src.nodata
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=nodata
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/statewide_rf.csv", index=False)

print(df)

date       state     1990-01     1990-02     1990-03    1990-04     1990-05  \
value  statewide  399.412593  244.831205  163.687242  73.548345  101.794447   

date     1990-06     1990-07    1990-08     1990-09  ...     2024-11  \
value  93.517066  127.565286  77.432914  176.091673  ...  165.368378   

date     2024-12     2025-01    2025-02     2025-03     2025-04    2025-05  \
value  31.055905  155.509047  31.366991  108.519044  105.936454  81.617498   

date    2025-06    2025-07    2025-08  
value  86.30062  78.660752  50.697025  

[1 rows x 429 columns]


In [ ]:
#Remove files
folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)